In [1]:
import duckdb
import pandas as pd

file = "'../data/process_data.json'"

exploded_events = duckdb.sql(f"""
        WITH unnested AS(
        SELECT Time as processing_time,
        unnest(ProcessData::JSON[]) as event
        FROM {file}
        )

        SELECT processing_time,
        event->>'$.Name' as name,
        event->>'$.Value' as value,
        event->>'$.Type' as type,
        event->>'$.Time' as event_time,
        event->>'$.Unit' as unit,
        FROM unnested
        """)

pivoted_events = duckdb.sql("""
        WITH pivoted AS(
        PIVOT exploded_events
        ON name
        USING FIRST(value)
        GROUP BY processing_time
        ORDER BY processing_time
        )
        SELECT CAST(processing_time AS TIMESTAMP_MS) as time,
        CAST("fAnlasstemperatur (Anlassen)" AS DOUBLE) as anlasstemp,
        CAST("fDrahtlaenge1 (Drehgeber)" AS DOUBLE) as drahtlaenge,
        CAST("fDurchmesser1 (Haerten)" AS DOUBLE) as durchmesser,
        CAST("fGeschwindigkeit1 (Drehgeber)" AS DOUBLE) as geschwindigkeit,
        CAST("fHaertetemperatur (Haerten)" AS DOUBLE) as haertetemp 
        FROM pivoted
        """)

In [2]:
temps_df = duckdb.sql("""
        SELECT
                time,
                anlasstemp,
                haertetemp
        FROM pivoted_events
        """).df()

In [3]:
def validate_alarms(df, window_minutes=5, min_count=10):
    """
    Validiert 3‑Sigma‑Alarme für Härte‑ und Anlasstemperatur.

    Parameter
    ---------
    df : pandas.DataFrame
        Muss die Spalten 'time' (datetime-ähnlich), 'haertetemp' und 'anlasstemp' enthalten.
    window_minutes : int
        Größe des gleitenden Zeitfensters in Minuten (Vorgabe: 5).
    min_count : int
        Mindestanzahl an Werten im Fenster, damit ein Alarm ausgelöst wird (Vorgabe: 10).

    Rückgabe
    --------
    pandas.DataFrame
        Enthält alle Alarme mit den Spalten:
        time, process, value, mean, std, upper, lower
    """
    # Zeitstempel normalisieren und sortieren
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time").set_index("time")

    alarms = []

    # Dictionary für die Prozessnamen (Spaltenname → Anzeigename)
    processes = {"haertetemp": "Haertetemperatur", "anlasstemp": "Anlasstemperatur"}

    for col, proc_name in processes.items():
        serie = df[col].dropna()  # fehlende Werte ignorieren

        # Gleitende Fensterberechnung (inklusive aktueller Zeile)
        rolling = serie.rolling(window=f"{window_minutes}min", min_periods=min_count)
        mean = rolling.mean()
        std = rolling.std(ddof=0)  # Grundgesamtheit (wie STDDEV_POP in Flink SQL)
        upper = mean + 3 * std
        lower = mean - 3 * std

        # Alarm, wenn Wert außerhalb der 3‑Sigma‑Grenzen
        alarm_condition = (serie > upper) | (serie < lower)
        alarm_indices = serie[alarm_condition].index

        for idx in alarm_indices:
            alarms.append(
                {
                    "time": idx,
                    "process": proc_name,
                    "value": serie.loc[idx],
                    "mean": mean.loc[idx],
                    "std": std.loc[idx],
                    "upper": upper.loc[idx],
                    "lower": lower.loc[idx],
                }
            )

    return pd.DataFrame(alarms).sort_values(by="time")

In [4]:
alarms = validate_alarms(temps_df, min_count=3000)

In [5]:
alarms[0:100]

,time,process,value,mean,std,upper,lower
77,2026-01-01 06:05:22.600,Anlasstemperatur,452.500,460.008867,1.974959,465.933742,454.083991
78,2026-01-01 06:05:30.700,Anlasstemperatur,466.300,460.011600,1.989805,465.981016,454.042184
79,2026-01-01 06:07:09.600,Anlasstemperatur,467.100,459.957300,2.011151,465.990752,453.923848
0,2026-01-01 06:07:30.400,Haertetemperatur,1018.040,988.490580,9.103937,1015.802391,961.178768
1,2026-01-01 06:07:37.200,Haertetemperatur,1017.405,988.815483,9.442757,1017.143754,960.487213
...,...,...,...,...,...,...,...
48,2026-01-01 06:31:38.200,Haertetemperatur,1011.671,987.158279,8.003195,1011.167864,963.148693
49,2026-01-01 06:31:38.500,Haertetemperatur,1014.734,987.176570,8.033788,1011.277933,963.075208
50,2026-01-01 06:31:39.900,Haertetemperatur,1012.738,987.247772,8.145583,1011.684521,962.811023
51,2026-01-01 06:31:40.000,Haertetemperatur,1017.495,987.256638,8.164005,1011.748651,962.764624


Auch ohne Spitzen liegen bei normalverteiltem Rauschen rund 0,3 % der Werte außerhalb von ±3σ. Bei 36.000 Messwerten pro Stunde sind das knapp 100 Alarme je Temperatur (abzüglich der ersten 5 Minuten, in denen das Fenster noch nicht voll ist), die eingebauten Spitzen kommen hinzu.

In [6]:
alarms.groupby("process").size()

process
Anlasstemperatur    91
Haertetemperatur    77
dtype: int64